# Ramsey detuning drift over a shift

Track how the detuning (qubit frequency - drive frequency) drifts across a series
of Ramsey experiments run over several hours — the signature of a TLS hop, a thermal
excursion, or flux-line crosstalk that the VLM is good at flagging.

Typical workflow: your control stack dumps one `.npz` per Ramsey run into a
watched directory; this notebook walks that directory, auto-fits each trace, and
plots the extracted detuning vs wall-clock time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
tau = np.linspace(0, 10e-6, 201)                 # 10 µs delay sweep
wall_times = np.arange(8) * 30 * 60               # every 30 min, 4 hours total

# Simulate detuning drifting from 120 kHz → 180 kHz with a TLS-like jump halfway.
detunings = np.array([120e3, 125e3, 132e3, 140e3, 170e3, 172e3, 175e3, 180e3])

for i, det in enumerate(detunings):
    phase = rng.uniform(0, 2*np.pi)
    y = 0.5 * np.exp(-tau/4e-6) * np.cos(2*np.pi*det*tau + phase) + 0.5
    y += rng.normal(0, 0.01, tau.shape)          # readout noise
    np.savez(f'ramsey_{i:02d}.npz', x=tau, y=y, wall_time=wall_times[i])

In [ ]:
from pathlib import Path
from qcal.data import from_npz

records = []
for path in sorted(Path('.').glob('ramsey_*.npz')):
    p = from_npz(path, experiment_type='ramsey', x_unit='s')
    if p.fit and p.fit.ok:
        det = next(v for k, v in p.fit.params.items() if k.startswith('detuning_per_'))
        t2s = next(v for k, v in p.fit.params.items() if k.startswith('t2star_'))
        records.append({'file': path.name, 'detuning_hz': det, 't2star_s': t2s,
                        'fit_quality': p.fit.fit_quality})

import pandas as pd
df = pd.DataFrame(records)
df

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(wall_times/3600, df['detuning_hz']/1e3, marker='o')
ax.set_xlabel('Wall time (hours)')
ax.set_ylabel('Fitted detuning (kHz)')
ax.set_title('Qubit detuning drift')
ax.grid(True, alpha=0.3)

Hand the **most recent** Ramsey trace to the VLM. It sees both the plot and the
fitted numbers in the prompt, so `notes` / `drift_prediction` usually flag the
step change in detuning.

In [ ]:
from qcal.analyzer import analyze_payload

latest = from_npz('ramsey_07.npz', experiment_type='ramsey', x_unit='s')
analyze_payload(latest, backend='auto')